In [ ]:
import pandas as pd
import numpy as np
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
df_csv = pd.read_csv('countries_of_the_world.csv')
print(df_csv.shape)
df_csv.head()

In [ ]:
df_txt = pd.read_csv('countries_of_the_world.txt', sep='\t')
print(df_txt.shape)
df_txt.head()

In [ ]:
with open('json_sample.json') as f:
    raw = json.load(f)

movies_dict = raw[0]
df_json = pd.DataFrame.from_dict(movies_dict, orient='index')
df_json.index.name = 'Title'
df_json = df_json.reset_index()
print(df_json.shape)
df_json.head()

In [ ]:
df_pop = pd.read_excel('world_population_excel_workbook.xlsx', sheet_name='world_population')
df_pop_subset = pd.read_excel('world_population_excel_workbook.xlsx', sheet_name='Sheet1')
print(df_pop.shape, df_pop_subset.shape)
df_pop.head()

,Rank,CCA3,Country,Capital,Continent,2022 Population,2020 Population,2015 Population,2010 Population,2000 Population,1990 Population,1980 Population,1970 Population,Area (kmÂ²),Density (per kmÂ²),Growth Rate,World Population Percentage
0,36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,1.0257,0.52
1,138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.9957,0.04
2,34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,1.0164,0.56
3,213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.9831,0.00
4,203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,1.0100,0.00


In [ ]:
for df in (df_csv, df_txt):
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

print("Missing values (csv):\n", df_csv.isna().sum())
print("\nDuplicate rows (csv):", df_csv.duplicated().sum())

print("\nMissing values (txt):\n", df_txt.isna().sum())
print("Duplicate rows (txt):", df_txt.duplicated().sum())

df_csv = df_csv.drop_duplicates().dropna()
df_txt = df_txt.drop_duplicates().dropna()

# Confirm both files describe the same dataset
print("\nCSV and TXT are identical:", df_csv.reset_index(drop=True).equals(df_txt.reset_index(drop=True)))

In [ ]:
# Standardize column names (case-insensitive merge of duplicate concept columns)
rename_map = {
    'popcornscore': 'Popcorn Score',
    'tomatoscore': 'Tomato Score',
    'rating': 'Rating',
}
df_json = df_json.rename(columns=rename_map)

# Merge duplicate columns caused by inconsistent casing (e.g. two 'Rating' columns)
df_json = df_json.T.groupby(level=0).first().T

# Replace known placeholder / invalid values with NaN
placeholders = ['unknown', 'unkown', 'unrated', -1]
df_json = df_json.replace(placeholders, np.nan)

# Clean Gross: strip $ and commas, convert to numeric
df_json['Gross'] = (
    df_json['Gross']
    .astype(str)
    .str.replace('[\$,]', '', regex=True)
    .replace('nan', np.nan)
)
df_json['Gross'] = pd.to_numeric(df_json['Gross'], errors='coerce')

# Coerce score/metascore columns to numeric
for col in ['Popcorn Score', 'Tomato Score', 'IMDB Metascore']:
    if col in df_json.columns:
        df_json[col] = pd.to_numeric(df_json[col], errors='coerce')

print("Missing values per column:\n", df_json.isna().sum())
print("\nDuplicate rows:", df_json.duplicated().sum())

df_json = df_json.drop_duplicates()
df_json.head(10)

### 2.3 Excel — World Population\n\nCheck for missing values and duplicate country rows.

In [ ]:
print("Missing values:\n", df_pop.isna().sum())
print("\nDuplicate rows:", df_pop.duplicated().sum())
print("Duplicate countries:", df_pop['Country'].duplicated().sum())

df_pop = df_pop.drop_duplicates()
df_pop.columns = df_pop.columns.str.strip()
df_pop.info()

## 3. Filtering Operations

### 3.1 Countries with a 2022 population over 100 million

In [ ]:
large_countries = df_pop[df_pop['2022 Population'] > 100_000_000] \
    .sort_values('2022 Population', ascending=False)[['Country', 'Continent', '2022 Population']]
large_countries

### 3.2 African countries with the highest population growth rate

In [ ]:
africa_growth = df_pop[df_pop['Continent'] == 'Africa'] \
    .sort_values('Growth Rate', ascending=False)[['Country', 'Growth Rate', '2022 Population']]
africa_growth.head(10)

### 3.3 Movies rated R with a Tomato Score above 85

In [ ]:
top_r_movies = df_json[(df_json['Rating'] == 'R') & (df_json['Tomato Score'] > 85)] \
    .sort_values('Tomato Score', ascending=False)[['Title', 'Rating', 'Tomato Score', 'Popcorn Score']]
top_r_movies

## 4. Group By / Summarization

### 4.1 Total & average 2022 population by continent

In [ ]:
continent_summary = df_pop.groupby('Continent').agg(
    total_population=('2022 Population', 'sum'),
    average_population=('2022 Population', 'mean'),
    country_count=('Country', 'count')
).sort_values('total_population', ascending=False)
continent_summary

### 4.2 Number of countries per world region (CSV/TXT dataset)

In [ ]:
region_counts = df_csv.groupby('Region')['Country'].count().sort_values(ascending=False)
region_counts

### 4.3 Average Tomato Score & Popcorn Score by movie Genre

In [ ]:
genre_summary = df_json.dropna(subset=['Genre']).groupby('Genre').agg(
    avg_tomato_score=('Tomato Score', 'mean'),
    avg_popcorn_score=('Popcorn Score', 'mean'),
    movie_count=('Title', 'count')
).sort_values('avg_tomato_score', ascending=False)
genre_summary

### 4.4 Average population growth rate by continent

In [ ]:
growth_by_continent = df_pop.groupby('Continent')['Growth Rate'].mean().sort_values(ascending=False)
growth_by_continent